# JBot Database Analytics Dashboard

This notebook provides comprehensive analytics for the `jbot.db` database, including:
- Database health and structure
- Player engagement and performance metrics
- Power-up usage and effectiveness
- Question usage patterns
- Streak and bonus analysis

## 1. Setup and Database Connection

This notebook assumes that `jbot.db` is in the same directory. Update the `db_file` variable below if needed.

In [1]:
import sqlite3
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

db_file = "jbot.db"

if not os.path.exists(db_file):
    print(
        f"Database file '{db_file}' not found. Please ensure it is in the same directory as this notebook."
    )
    conn = None
else:
    conn = sqlite3.connect(db_file)
    print("✓ Successfully connected to the database.")

✓ Successfully connected to the database.


## 2. Database Size and Record Counts

This section provides a high-level overview of the database's size on disk and the number of records in each table.


In [2]:
if "conn" in locals() and conn:
    # Get database file size
    db_size = os.path.getsize(db_file)
    print(
        f"Database file size: {db_size / 1024:.2f} KB ({db_size / (1024*1024):.2f} MB)"
    )

    # Get record counts for each table
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    record_counts = []
    for table_name in tables:
        table_name = table_name[0]
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cursor.fetchone()[0]
        record_counts.append({"Table": table_name, "Record Count": count})

    df_counts = pd.DataFrame(record_counts)
    df_counts = df_counts.sort_values("Record Count", ascending=False)
    print("\n📊 Record counts per table:")
    print(df_counts.to_string(index=False))

Database file size: 1060.00 KB (1.04 MB)

📊 Record counts per table:
              Table  Record Count
            guesses          3998
           messages          1520
daily_player_states          1262
    daily_questions           202
          questions           184
      powerup_usage            67
  score_adjustments            55
            players            17
    sqlite_sequence             9
alternative_answers             4
              roles             2
        subscribers             1
       player_roles             1
            seasons             0
      season_scores             0
  season_challenges             0
season_daily_scores             0


## 3. Player Engagement & Performance Metrics

Comprehensive analysis of player activity, accuracy, and competitive performance.

In [3]:
if "conn" in locals() and conn:
    # Top Players by Score
    query = """
    SELECT
        p.name,
        p.score,
        p.answer_streak as current_streak,
        COUNT(DISTINCT CASE WHEN g.is_correct = 1 THEN g.daily_question_id END) as correct_answers,
        COUNT(DISTINCT g.daily_question_id) as days_participated,
        ROUND(100.0 * COUNT(CASE WHEN g.is_correct = 1 THEN 1 END) / COUNT(g.id), 1) as accuracy_pct
    FROM players p
    LEFT JOIN guesses g ON p.id = g.player_id
    GROUP BY p.id, p.name, p.score, p.answer_streak
    ORDER BY p.score DESC
    LIMIT 15;
    """
    df_leaderboard = pd.read_sql_query(query, conn)
    print("🏆 Top Players Leaderboard")
    print(df_leaderboard.to_string(index=False))

    # Visualize top 10 by score
    fig = px.bar(
        df_leaderboard.head(10),
        x="name",
        y="score",
        title="Top 10 Players by Score",
        labels={"score": "Total Score", "name": "Player"},
        color="score",
        color_continuous_scale="Viridis",
    )
    fig.show()

    # Player Accuracy Distribution
    query = """
    SELECT
        p.name,
        ROUND(100.0 * COUNT(CASE WHEN g.is_correct = 1 THEN 1 END) / COUNT(g.id), 1) as accuracy
    FROM players p
    INNER JOIN guesses g ON p.id = g.player_id
    GROUP BY p.id, p.name
    HAVING COUNT(g.id) >= 5
    ORDER BY accuracy DESC;
    """
    df_accuracy = pd.read_sql_query(query, conn)

    fig = px.histogram(
        df_accuracy,
        x="accuracy",
        nbins=20,
        title="Player Accuracy Distribution (min 5 guesses)",
        labels={"accuracy": "Accuracy %", "count": "Number of Players"},
    )
    fig.show()

    # Active Player Trends (last 30 days)
    query = """
    SELECT
        DATE(g.guessed_at) as date,
        COUNT(DISTINCT g.player_id) as active_players
    FROM guesses g
    WHERE g.guessed_at >= DATE('now', '-30 days')
    GROUP BY DATE(g.guessed_at)
    ORDER BY date;
    """
    df_active = pd.read_sql_query(query, conn)
    if not df_active.empty:
        df_active["date"] = pd.to_datetime(df_active["date"])
        fig = px.line(
            df_active,
            x="date",
            y="active_players",
            title="Daily Active Players (Last 30 Days)",
            labels={"active_players": "Active Players", "date": "Date"},
        )
        fig.show()

    # Total Participation Summary
    query = """
    SELECT
        COUNT(DISTINCT player_id) as total_players,
        COUNT(DISTINCT CASE WHEN guessed_at >= DATE('now', '-7 days') THEN player_id END) as active_last_7d,
        COUNT(DISTINCT CASE WHEN guessed_at >= DATE('now', '-30 days') THEN player_id END) as active_last_30d
    FROM guesses;
    """
    df_summary = pd.read_sql_query(query, conn)
    print("\n📈 Player Activity Summary:")
    print(f"Total players with guesses: {df_summary.iloc[0]['total_players']}")
    print(f"Active in last 7 days: {df_summary.iloc[0]['active_last_7d']}")
    print(f"Active in last 30 days: {df_summary.iloc[0]['active_last_30d']}")

🏆 Top Players Leaderboard
            name  score  current_streak  correct_answers  days_participated  accuracy_pct
           lanie  19689              10              141                148          47.2
           Blake  19155               9              151                166          32.0
           Kenny  16167               3              122                145          22.9
         Kristen  15897               6              121                137          33.0
         TaoMike  15361               4              104                113          40.9
        Kimberly  13987               1              100                110          28.8
    bunnyfarts83  13311               0               91                108          25.9
       Juniornlx  12600               1               99                127          33.4
      DrKnickers  11405               0               85                 96          36.5
        Jonathan  10960               0               92                10


📈 Player Activity Summary:
Total players with guesses: 17
Active in last 7 days: 14
Active in last 30 days: 15


## 4. Power-Up Usage & Effectiveness

Analysis of power-up mechanics, including usage patterns and streak performance.

In [4]:
if "conn" in locals() and conn:
    # Power-up usage by type
    query = """
    SELECT
        powerup_type,
        COUNT(*) as usage_count,
        COUNT(DISTINCT user_id) as unique_users
    FROM powerup_usage
    GROUP BY powerup_type
    ORDER BY usage_count DESC;
    """
    df_powerups = pd.read_sql_query(query, conn)

    if not df_powerups.empty:
        print("⚡ Power-Up Usage Statistics:")
        print(df_powerups.to_string(index=False))

        # Visualize power-up usage
        fig = px.bar(
            df_powerups,
            x="powerup_type",
            y="usage_count",
            title="Power-Up Usage by Type",
            labels={"powerup_type": "Power-Up Type", "usage_count": "Times Used"},
            color="usage_count",
            color_continuous_scale="Plasma",
        )
        fig.show()

        # Power-up usage over time
        query = """
        SELECT
            DATE(used_at) as date,
            powerup_type,
            COUNT(*) as count
        FROM powerup_usage
        WHERE used_at >= DATE('now', '-30 days')
        GROUP BY DATE(used_at), powerup_type
        ORDER BY date;
        """
        df_powerup_trends = pd.read_sql_query(query, conn)
        if not df_powerup_trends.empty:
            df_powerup_trends["date"] = pd.to_datetime(df_powerup_trends["date"])
            fig = px.line(
                df_powerup_trends,
                x="date",
                y="count",
                color="powerup_type",
                title="Power-Up Usage Trends (Last 30 Days)",
                labels={
                    "count": "Usage Count",
                    "date": "Date",
                    "powerup_type": "Power-Up",
                },
            )
            fig.show()

        # Top power-up users
        query = """
        SELECT
            p.name,
            pu.powerup_type,
            COUNT(*) as times_used
        FROM powerup_usage pu
        JOIN players p ON pu.user_id = p.id
        GROUP BY pu.user_id, p.name, pu.powerup_type
        ORDER BY times_used DESC
        LIMIT 10;
        """
        df_top_users = pd.read_sql_query(query, conn)
        print("\n🎯 Top Power-Up Users:")
        print(df_top_users.to_string(index=False))
    else:
        print("No power-up usage data found.")

    # Streak Analysis
    print("\n\n🔥 Answer Streak Analysis:")
    query = """
    SELECT
        p.name,
        p.answer_streak as current_streak,
        p.score
    FROM players p
    WHERE p.answer_streak > 0
    ORDER BY p.answer_streak DESC
    LIMIT 10;
    """
    df_streaks = pd.read_sql_query(query, conn)
    if not df_streaks.empty:
        print("Top Current Streaks:")
        print(df_streaks.to_string(index=False))

        # Streak distribution
        query = """
        SELECT
            answer_streak,
            COUNT(*) as player_count
        FROM players
        WHERE answer_streak > 0
        GROUP BY answer_streak
        ORDER BY answer_streak;
        """
        df_streak_dist = pd.read_sql_query(query, conn)
        fig = px.bar(
            df_streak_dist,
            x="answer_streak",
            y="player_count",
            title="Active Streak Distribution",
            labels={
                "answer_streak": "Streak Length",
                "player_count": "Number of Players",
            },
        )
        fig.show()
    else:
        print("No active streaks found.")

    # Shield usage
    query = """
    SELECT
        COUNT(*) as total_shields_active
    FROM players
    WHERE active_shield = 1;
    """
    df_shields = pd.read_sql_query(query, conn)
    print(
        f"\n🛡️ Currently active shields: {df_shields.iloc[0]['total_shields_active']}"
    )

⚡ Power-Up Usage Statistics:
powerup_type  usage_count  unique_users
        jinx           28             5
       steal           22             6
      shield           11             2
        rest            6             5



🎯 Top Power-Up Users:
    name powerup_type  times_used
   Kenny        steal          10
   lanie       shield          10
   Blake         jinx           9
   lanie         jinx           7
Jonathan         jinx           6
   Kenny         jinx           5
   lanie        steal           4
Jonathan        steal           2
   Blake         rest           2
   Blake        steal           2


🔥 Answer Streak Analysis:
Top Current Streaks:
            name  current_streak  score
           lanie              10  19689
           Blake               9  19155
         Kristen               6  15897
         TaoMike               4  15361
           Kenny               3  16167
israelromero3102               2  10248
       Juniornlx               1  12600
        Kimberly               1  13987
          Shiver               1   7239



🛡️ Currently active shields: 0


## 5. Question Usage & Dataset Analysis

Analysis of trivia questions, including usage rates by source dataset and difficulty patterns.

In [5]:
if "conn" in locals() and conn:
    # Total questions vs. Used questions
    query_total = "SELECT COUNT(*) FROM questions;"
    total_questions = pd.read_sql_query(query_total, conn).iloc[0, 0]

    query_used = "SELECT COUNT(DISTINCT question_id) FROM daily_questions;"
    used_questions = pd.read_sql_query(query_used, conn).iloc[0, 0]

    print(f"📚 Question Pool Statistics:")
    print(f"Total questions available: {total_questions}")
    print(f"Unique questions used: {used_questions}")
    if total_questions > 0:
        print(f"Usage percentage: {used_questions / total_questions * 100:.1f}%")
        print(f"Remaining unused: {total_questions - used_questions}")

    # Question usage by source with enhanced metrics
    query = """
    SELECT
        q.source,
        COUNT(q.id) as total_questions,
        COUNT(DISTINCT dq.question_id) as used_questions,
        ROUND(AVG(q.value), 1) as avg_value,
        COUNT(DISTINCT CASE WHEN q.hint_text IS NOT NULL THEN q.id END) as questions_with_hints
    FROM questions q
    LEFT JOIN daily_questions dq ON q.id = dq.question_id
    GROUP BY q.source
    ORDER BY total_questions DESC;
    """
    df_source = pd.read_sql_query(query, conn)
    df_source["usage_pct"] = (
        df_source["used_questions"] / df_source["total_questions"] * 100
    ).round(1)
    df_source["hint_pct"] = (
        df_source["questions_with_hints"] / df_source["total_questions"] * 100
    ).round(1)

    print("\n📊 Question Usage by Source Dataset:")
    print(df_source.to_string(index=False))

    # Visualize usage by source
    fig = px.bar(
        df_source,
        x="source",
        y=["used_questions", "total_questions"],
        title="Question Usage by Source Dataset",
        labels={
            "value": "Number of Questions",
            "variable": "Status",
            "source": "Dataset",
        },
        barmode="group",
    )
    fig.show()

    # Question difficulty analysis (if value represents difficulty)
    query = """
    SELECT
        CASE
            WHEN q.value <= 100 THEN 'Easy'
            WHEN q.value <= 200 THEN 'Medium'
            ELSE 'Hard'
        END as difficulty,
        COUNT(*) as question_count,
        COUNT(DISTINCT dq.id) as times_used
    FROM questions q
    LEFT JOIN daily_questions dq ON q.id = dq.question_id
    WHERE q.value IS NOT NULL
    GROUP BY difficulty
    ORDER BY
        CASE difficulty
            WHEN 'Easy' THEN 1
            WHEN 'Medium' THEN 2
            ELSE 3
        END;
    """
    df_difficulty = pd.read_sql_query(query, conn)
    if not df_difficulty.empty:
        fig = px.pie(
            df_difficulty,
            names="difficulty",
            values="question_count",
            title="Question Distribution by Difficulty (based on value)",
            color_discrete_sequence=px.colors.sequential.RdBu,
        )
        fig.show()

    # Recently used questions
    query = """
    SELECT
        q.question_text,
        q.source,
        q.value,
        dq.sent_at
    FROM daily_questions dq
    JOIN questions q ON dq.question_id = q.id
    ORDER BY dq.sent_at DESC
    LIMIT 10;
    """
    df_recent = pd.read_sql_query(query, conn)
    print("\n📅 Recently Used Questions:")
    df_recent["question_preview"] = df_recent["question_text"].str[:60] + "..."
    print(
        df_recent[["question_preview", "source", "value", "sent_at"]].to_string(
            index=False
        )
    )

📚 Question Pool Statistics:
Total questions available: 184
Unique questions used: 170
Usage percentage: 92.4%
Remaining unused: 14

📊 Question Usage by Source Dataset:
                                       source  total_questions  used_questions  avg_value  questions_with_hints  usage_pct  hint_pct
                           Riddles with Hints               62              35      100.0                    30       56.5      48.4
                                   5th Grader               62              60      100.0                    62       96.8     100.0
                                          NaN               23              18      100.0                    16       78.3      69.6
                                    Jeopardy!               19              19      178.9                    19      100.0     100.0
          Are You Smarter Than a Fifth Grader               17              17      100.0                    17      100.0     100.0
                                ge


📅 Recently Used Questions:
                                               question_preview                              source  value    sent_at
Which archetype aids the hero in a play,but doesn’t start an... Are You Smarter Than a Fifth Grader    100 2026-03-07
\"The Marine Metropolis of Virginia" & "Vacationland, U.S.A....                           Jeopardy!    200 2026-03-06
This traditional fuel of the British Isles is still hand cut...                           Jeopardy!    200 2026-03-05
                What Jewish holiday is the Day of Atonement?... Are You Smarter Than a Fifth Grader    100 2026-03-04
                   What country borders Finland to the west?... Are You Smarter Than a Fifth Grader    100 2026-03-03
  Its state flower is the Carolina Jessamine — take a guess!...                           Jeopardy!    200 2026-03-02
I carry a hundred eyes on my train, but cannot see where I h...                         gemini_hard    300 2026-03-01
A swift fellow, this Austria

## 6. Guess Patterns & Timing Analysis

Analysis of when and how players submit their guesses.

In [6]:
if "conn" in locals() and conn:
    # Overall guess statistics
    query = """
    SELECT
        COUNT(*) as total_guesses,
        COUNT(CASE WHEN is_correct = 1 THEN 1 END) as correct_guesses,
        COUNT(CASE WHEN is_correct = 0 THEN 1 END) as incorrect_guesses,
        ROUND(100.0 * COUNT(CASE WHEN is_correct = 1 THEN 1 END) / COUNT(*), 1) as overall_accuracy
    FROM guesses;
    """
    df_stats = pd.read_sql_query(query, conn)
    print("🎯 Overall Guess Statistics:")
    print(f"Total guesses: {df_stats.iloc[0]['total_guesses']}")
    print(f"Correct: {df_stats.iloc[0]['correct_guesses']}")
    print(f"Incorrect: {df_stats.iloc[0]['incorrect_guesses']}")
    print(f"Overall accuracy: {df_stats.iloc[0]['overall_accuracy']}%")

    # Guesses per day distribution
    query = """
    SELECT
        daily_question_id,
        COUNT(*) as guess_count,
        COUNT(CASE WHEN is_correct = 1 THEN 1 END) as correct_count
    FROM guesses
    GROUP BY daily_question_id
    ORDER BY daily_question_id DESC
    LIMIT 30;
    """
    df_daily_guesses = pd.read_sql_query(query, conn)
    if not df_daily_guesses.empty:
        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=df_daily_guesses["daily_question_id"],
                y=df_daily_guesses["guess_count"],
                name="Total Guesses",
                mode="lines+markers",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df_daily_guesses["daily_question_id"],
                y=df_daily_guesses["correct_count"],
                name="Correct Guesses",
                mode="lines+markers",
            )
        )
        fig.update_layout(
            title="Guess Activity per Question (Last 30)",
            xaxis_title="Daily Question ID",
            yaxis_title="Number of Guesses",
        )
        fig.show()

    # Guess timing analysis (hour of day) - converted to Pacific time (handles DST)
    query = """
    SELECT
        guessed_at,
        is_correct
    FROM guesses;
    """
    df_guesses_raw = pd.read_sql_query(query, conn)
    if not df_guesses_raw.empty:
        df_guesses_raw["guessed_at"] = pd.to_datetime(
            df_guesses_raw["guessed_at"], utc=True
        )
        df_guesses_raw["guessed_at_pt"] = df_guesses_raw["guessed_at"].dt.tz_convert(
            "America/Los_Angeles"
        )
        df_guesses_raw["hour"] = df_guesses_raw["guessed_at_pt"].dt.hour

        df_hourly = (
            df_guesses_raw.groupby("hour")
            .agg(
                guess_count=("is_correct", "count"),
                correct_count=("is_correct", "sum"),
            )
            .reset_index()
        )
        df_hourly["accuracy"] = (
            df_hourly["correct_count"] / df_hourly["guess_count"] * 100
        ).round(1)

        fig = go.Figure()
        fig.add_trace(
            go.Bar(
                x=df_hourly["hour"],
                y=df_hourly["guess_count"],
                name="Guess Count",
                yaxis="y",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df_hourly["hour"],
                y=df_hourly["accuracy"],
                name="Accuracy %",
                yaxis="y2",
                mode="lines+markers",
                line=dict(color="red"),
            )
        )
        fig.update_layout(
            title="Guess Activity by Hour of Day (Pacific Time)",
            xaxis_title="Hour (Pacific Time, 24-hour format)",
            yaxis=dict(title="Guess Count"),
            yaxis2=dict(title="Accuracy %", overlaying="y", side="right"),
            hovermode="x unified",
        )
        fig.show()

    # Multiple guess analysis
    query = """
    SELECT
        player_id,
        daily_question_id,
        COUNT(*) as attempts
    FROM guesses
    GROUP BY player_id, daily_question_id
    HAVING COUNT(*) > 1
    ORDER BY attempts DESC
    LIMIT 10;
    """
    df_multi = pd.read_sql_query(query, conn)
    if not df_multi.empty:
        print("\n🔄 Top instances of multiple guesses on same question:")
        print(df_multi.to_string(index=False))

🎯 Overall Guess Statistics:
Total guesses: 3998.0
Correct: 1335.0
Incorrect: 2663.0
Overall accuracy: 33.4%



🔄 Top instances of multiple guesses on same question:
          player_id  daily_question_id  attempts
 884572470995189762                 75        36
 333686707260096515                 75        30
1437645597137178695                166        25
1437645597137178695                191        22
 333686707260096515                 63        21
 884572470995189762                170        20
 196397949893345281                183        19
1418068935911145516                 91        18
1418068935911145516                142        18
1418068935911145516                126        17


## 7. Data Integrity & Sanity Checks

Verification checks for database consistency and potential issues.

In [7]:
if "conn" in locals() and conn:
    print("🔍 Database Integrity Checks:\n")

    # Check for orphaned records
    query = """
    SELECT COUNT(*) as orphaned_guesses
    FROM guesses g
    LEFT JOIN players p ON g.player_id = p.id
    WHERE p.id IS NULL;
    """
    orphaned = pd.read_sql_query(query, conn).iloc[0, 0]
    if orphaned > 0:
        print(f"⚠️  Warning: {orphaned} orphaned guesses (player not found)")
    else:
        print("✓ No orphaned guesses")

    # Check for questions without answers
    query = """
    SELECT COUNT(*) as missing_answers
    FROM questions
    WHERE answer_text IS NULL OR answer_text = '';
    """
    missing = pd.read_sql_query(query, conn).iloc[0, 0]
    if missing > 0:
        print(f"⚠️  Warning: {missing} questions missing answers")
    else:
        print("✓ All questions have answers")

    # Daily question continuity check
    query = """
    SELECT
        MIN(DATE(sent_at)) as first_question,
        MAX(DATE(sent_at)) as last_question,
        COUNT(DISTINCT DATE(sent_at)) as unique_days
    FROM daily_questions;
    """
    df_dates = pd.read_sql_query(query, conn)
    if not df_dates.empty and df_dates.iloc[0]["first_question"]:
        first = pd.to_datetime(df_dates.iloc[0]["first_question"])
        last = pd.to_datetime(df_dates.iloc[0]["last_question"])
        expected_days = (last - first).days + 1
        actual_days = df_dates.iloc[0]["unique_days"]

        print(f"\n📅 Daily Questions Timeline:")
        print(f"First question: {first.strftime('%Y-%m-%d')}")
        print(f"Last question: {last.strftime('%Y-%m-%d')}")
        print(f"Expected days: {expected_days}")
        print(f"Actual days with questions: {actual_days}")

        if expected_days != actual_days:
            print(f"⚠️  Warning: {expected_days - actual_days} days missing questions")

            # Find missing dates
            query = """
            SELECT DATE(sent_at) as date
            FROM daily_questions
            ORDER BY sent_at;
            """
            df_all_dates = pd.read_sql_query(query, conn)
            date_range = pd.date_range(start=first, end=last)
            existing_dates = pd.to_datetime(df_all_dates["date"])
            missing_dates = date_range.difference(existing_dates)

            if len(missing_dates) > 0 and len(missing_dates) <= 10:
                print("\nMissing dates:")
                for date in missing_dates:
                    print(f"  - {date.strftime('%Y-%m-%d')}")
            elif len(missing_dates) > 10:
                print(f"\n  ({len(missing_dates)} dates missing - too many to list)")
        else:
            print("✓ No missing days")

    # Player role distribution
    query = """
    SELECT
        COALESCE(r.name, 'No Role') as role_name,
        COUNT(DISTINCT COALESCE(pr.player_id, p.id)) as player_count
    FROM players p
    LEFT JOIN player_roles pr ON p.id = pr.player_id
    LEFT JOIN roles r ON pr.role_id = r.id
    GROUP BY r.name;
    """
    df_roles = pd.read_sql_query(query, conn)
    if not df_roles.empty:
        print("\n👥 Player Role Distribution:")
        print(df_roles.to_string(index=False))

        fig = px.pie(
            df_roles,
            names="role_name",
            values="player_count",
            title="Player Role Distribution",
        )
        fig.show()

    # Score adjustment audit
    query = """
    SELECT COUNT(*) as adjustment_count
    FROM score_adjustments;
    """
    adjustments = pd.read_sql_query(query, conn).iloc[0, 0]
    if adjustments > 0:
        print(f"\n📝 Total manual score adjustments: {adjustments}")

        query = """
        SELECT
            p.name as player,
            SUM(sa.amount) as total_adjustment
        FROM score_adjustments sa
        JOIN players p ON sa.player_id = p.id
        GROUP BY sa.player_id, p.name
        ORDER BY total_adjustment DESC
        LIMIT 5;
        """
        df_adj = pd.read_sql_query(query, conn)
        print("\nTop adjusted players:")
        print(df_adj.to_string(index=False))

    # Alternative answers
    query = """
    SELECT COUNT(DISTINCT question_id) as questions_with_alts
    FROM alternative_answers;
    """
    alt_count = pd.read_sql_query(query, conn).iloc[0, 0]
    print(f"\n✏️  Questions with alternative answers: {alt_count}")

    print("\n✅ Integrity check complete!")

🔍 Database Integrity Checks:

✓ No orphaned guesses
✓ All questions have answers

📅 Daily Questions Timeline:
First question: 2025-09-15
Last question: 2026-03-07
Expected days: 174
Actual days with questions: 170
⚠️  Warning: 4 days missing questions

Missing dates:
  - 2025-09-16
  - 2025-10-09
  - 2025-10-10
  - 2025-12-22

👥 Player Role Distribution:
  role_name  player_count
    No Role            16
first place             1



📝 Total manual score adjustments: 55

Top adjusted players:
      player  total_adjustment
    Jonathan               540
       Kenny               445
       lanie               439
bunnyfarts83               410
  DrKnickers               351

✏️  Questions with alternative answers: 4

✅ Integrity check complete!


## 8. Advanced Metrics & Insights

Deeper analysis combining multiple data sources for strategic insights.

In [8]:
if "conn" in locals() and conn:
    # Player engagement score (combines participation, accuracy, and consistency)
    query = """
    SELECT
        p.name as player_name,
        COUNT(DISTINCT g.daily_question_id) as days_played,
        ROUND(100.0 * COUNT(CASE WHEN g.is_correct = 1 THEN 1 END) / COUNT(g.id), 1) as accuracy,
        p.score as total_score,
        p.answer_streak as streak,
        (COUNT(DISTINCT g.daily_question_id) *
         (100.0 * COUNT(CASE WHEN g.is_correct = 1 THEN 1 END) / COUNT(g.id)) / 100.0 *
         (1 + p.answer_streak * 0.1)) as engagement_score
    FROM players p
    JOIN guesses g ON p.id = g.player_id
    GROUP BY p.id, p.name, p.score, p.answer_streak
    HAVING COUNT(g.id) >= 5
    ORDER BY engagement_score DESC
    LIMIT 15;
    """
    df_engagement = pd.read_sql_query(query, conn)
    print("🎖️ Player Engagement Rankings (participation × accuracy × streak):")
    print(df_engagement.to_string(index=False))

    # Question difficulty vs participation
    query = """
    SELECT
        q.source as dataset_source,
        ROUND(AVG(q.value), 1) as avg_difficulty,
        COUNT(DISTINCT dq.id) as times_asked,
        ROUND(AVG(guess_counts.total_guesses), 1) as avg_guesses_per_question,
        ROUND(AVG(guess_counts.correct_rate), 1) as avg_correct_rate
    FROM questions q
    JOIN daily_questions dq ON q.id = dq.question_id
    JOIN (
        SELECT
            daily_question_id,
            COUNT(*) as total_guesses,
            100.0 * COUNT(CASE WHEN is_correct = 1 THEN 1 END) / COUNT(*) as correct_rate
        FROM guesses
        GROUP BY daily_question_id
    ) guess_counts ON dq.id = guess_counts.daily_question_id
    GROUP BY q.source
    ORDER BY avg_difficulty DESC;
    """
    df_difficulty_engagement = pd.read_sql_query(query, conn)
    if not df_difficulty_engagement.empty:
        # Fill NaNs in dataset_source to prevent plot errors
        df_difficulty_engagement["dataset_source"] = df_difficulty_engagement[
            "dataset_source"
        ].fillna("Unknown")

        print("\n📊 Dataset Difficulty vs Player Engagement:")
        print(df_difficulty_engagement.to_string(index=False))

        fig = px.scatter(
            df_difficulty_engagement,
            x="avg_difficulty",
            y="avg_correct_rate",
            size="times_asked",
            color="dataset_source",
            hover_data=["avg_guesses_per_question"],
            title="Question Difficulty vs Success Rate by Source",
            labels={
                "avg_difficulty": "Average Question Value",
                "avg_correct_rate": "Average Correct Rate %",
                "dataset_source": "Dataset",
            },
        )
        fig.show()

    # Power-up effectiveness analysis
    query = """
    SELECT
        pu.powerup_type,
        COUNT(DISTINCT pu.user_id) as unique_users,
        p_stats.avg_score_before as avg_score_before,
        p_stats.avg_score_after as avg_score_after
    FROM powerup_usage pu
    JOIN (
        SELECT
            pu2.user_id,
            pu2.powerup_type,
            AVG(CASE WHEN dps.snapshot_at < pu2.used_at THEN dps.score END) as avg_score_before,
            AVG(CASE WHEN dps.snapshot_at >= pu2.used_at THEN dps.score END) as avg_score_after
        FROM powerup_usage pu2
        JOIN daily_player_states dps ON pu2.user_id = dps.player_id
        GROUP BY pu2.user_id, pu2.powerup_type
    ) p_stats ON pu.user_id = p_stats.user_id AND pu.powerup_type = p_stats.powerup_type
    GROUP BY pu.powerup_type;
    """
    df_powerup_effect = pd.read_sql_query(query, conn)
    if not df_powerup_effect.empty and not df_powerup_effect.isnull().all().all():
        print("\n⚡ Power-Up Effectiveness Analysis:")
        print(df_powerup_effect.to_string(index=False))

    # Cohort retention analysis (players by join week)
    query = """
    SELECT
        strftime('%Y-W%W', p.created_at) as join_week,
        COUNT(DISTINCT p.id) as players_joined,
        COUNT(DISTINCT CASE
            WHEN g.guessed_at >= DATE('now', '-7 days') THEN p.id
        END) as still_active
    FROM players p
    LEFT JOIN guesses g ON p.id = g.player_id
    WHERE p.created_at >= DATE('now', '-90 days')
    GROUP BY join_week
    ORDER BY join_week;
    """
    df_cohort = pd.read_sql_query(query, conn)
    if not df_cohort.empty:
        df_cohort["retention_rate"] = (
            df_cohort["still_active"] / df_cohort["players_joined"] * 100
        ).round(1)

        print("\n📈 Player Retention by Cohort (Last 90 days):")
        print(df_cohort.to_string(index=False))

        fig = go.Figure()
        fig.add_trace(
            go.Bar(
                x=df_cohort["join_week"],
                y=df_cohort["players_joined"],
                name="Players Joined",
                yaxis="y",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df_cohort["join_week"],
                y=df_cohort["retention_rate"],
                name="Retention Rate %",
                yaxis="y2",
                mode="lines+markers",
                line=dict(color="green"),
            )
        )
        fig.update_layout(
            title="Player Retention by Join Week",
            xaxis_title="Join Week",
            yaxis=dict(title="Players Joined"),
            yaxis2=dict(title="Retention Rate %", overlaying="y", side="right"),
            hovermode="x unified",
        )
        fig.show()

🎖️ Player Engagement Rankings (participation × accuracy × streak):
     player_name  days_played  accuracy  total_score  streak  engagement_score
           lanie          148      47.2        19689      10        139.750820
           Blake          166      32.0        19155       9        100.797938
         Kristen          137      33.0        15897       6         72.295515
         TaoMike          113      40.9        15361       4         64.634241
       Juniornlx          127      33.4        12600       1         46.722408
           Kenny          145      22.9        16167       3         43.256530
        Jonathan          108      37.3        10960       0         40.285714
      DrKnickers           96      36.5        11405       0         35.021459
        Kimberly          110      28.8        13987       1         34.817664
israelromero3102           77      36.9        10248       2         34.069274
    bunnyfarts83          108      25.9        13311       0    


⚡ Power-Up Effectiveness Analysis:
powerup_type  unique_users  avg_score_before  avg_score_after
        jinx             5       6143.655172      9042.839117
        rest             5      12783.146667     18889.500000
      shield             2               NaN     12941.753247
       steal             6       8618.478261     13700.801527

📈 Player Retention by Cohort (Last 90 days):
join_week  players_joined  still_active  retention_rate
 2025-W49               1             0             0.0
 2026-W02               2             2           100.0


## 9. Cleanup

Close the database connection.

In [9]:
if "conn" in locals() and conn:
    conn.close()
    print("✓ Database connection closed successfully.")

✓ Database connection closed successfully.
